### ray basics

- 在单机写 Python 时，我们用 threading 或 multiprocessing 来做并发。但如果你的任务需要跨越 10 台机器、80 张显卡、数百个 CPU 核心呢？传统的做法是写复杂的 RPC（如 gRPC）、消息队列（RabbitMQ）或者用 MPI，极其痛苦。
- Ray 是一个开源的分布式计算框架。它的核心理念是：让你用写单机 Python 代码的方式，无缝写出能在成百上千台机器上运行的分布式代码。
- Ray 的两个核心概念
    - Task（任务 / 无状态）：给一个普通的 Python 函数加上 @ray.remote 装饰器，当调用 func.remote() 时，Ray 会自动把这个函数扔到集群里某台有空闲 CPU/GPU 的机器上去异步执行，并立刻返回一个 Future（凭证）。
    - Actor（角色 / 有状态）：给一个 Python 类 加上 @ray.remote，Ray 会在集群的某个进程里实例化这个类。这个实例是常驻的、有状态的。可以跨机器调用它的方法，它内部的变量（比如计数器、连接池）会一直保持。

### why ray in RL framework

> ray 作为分布式操作系统

- 资源抽象与物理隔离（Resource Management）
    - Ray 允许你给 Python 类打上标签，比如 @ray.remote(num_gpus=1) 或 @ray.remote(num_cpus=2)。 在 ReTool 中：
        - Trainer 和 Rollout 被 Ray 调度到 GPU 上。
        - AgentLoopWorker 和 ExecutionWorker 被 Ray 调度到 CPU 节点上。 必要性：实现了真正的物理隔离。GPU 专心做矩阵乘法，CPU 专心处理 Sandbox 的网络请求和字符串解析，互不干扰。
- Actor 模型：常驻内存的服务化（Stateful Microservices）
    - 在 RL 中，模型权重非常庞大（几十 GB），不可能每次生成或训练时重新加载。 Ray 的 Actor 机制允许把 vLLM 引擎和 FSDP 引擎变成常驻内存的微服务。 必要性：verl 利用 Ray Actor，让 vLLM 进程和 FSDP 进程在不同的显存区域（甚至不同的机器上）一直活着。需要生成时，RPC 调用 vLLM Actor；需要训练时，RPC 调用 FSDP Actor。
- 分布式对象存储与零拷贝（Plasma Object Store）
    - 生成的数据（Trajectories）和更新后的权重（Weights）需要在不同的 Actor 之间传递。 必要性：Ray 底层有一个极速的共享内存对象存储。当 vLLM 生成了上万个 Token，它不需要通过低效的 Python 序列化传给 Trainer，而是直接放在 Ray 的共享内存里，Trainer 可以以接近“零拷贝”的速度读取。

- Actor Model（生成与训练）：需要极高的 GPU 算力，通常需要张量并行（TP）跨多卡运行（比如用 vLLM 生成，用 Megatron/FSDP 训练）。
- Reference Model（参考模型）：只需要做前向推理，算 KL 散度。
- Reward Model（奖励模型）：也是前向推理，给生成的答案打分。
- Environment / Rollout（环境交互/工具调用）：纯 CPU 和 I/O 密集型任务。比如解析模型的输出、调用外部 API（Sandbox 执行代码）、跑 Agent 状态机。

如果不用 Ray： 需要把这些模块塞进同一个 Python 进程里。结果就是：GPU 在等 CPU 解析字符串、等网络请求（Sandbox），导致昂贵的 GPU 处于 0% 利用率；而且显存会被各个模型互相挤占（OOM）。

引入 Ray 之后（verl 的架构）： Ray 像一个大管家。它把 vLLM 引擎包装成 Ray Actor 放在 GPU 节点上；把 AgentLoopWorker 和 ExecutionWorker 包装成 Ray Actor 放在纯 CPU 节点（或空闲的 CPU 核心）上。

- GPU 只管生成文本。
- 生成完，通过 Ray 把文本传给 CPU 上的 AgentLoopWorker。
- CPU 慢慢去请求 Sandbox、执行代码、计算 Reward。
- 算完再通过 Ray 把 Reward 传回给 GPU 更新权重。 这就是为什么 verl 能做到极高吞吐量的原因：异构计算资源的解耦。

### retool of verl

```
[Driver]
  fit() loop
  └── async_rollout_manager.generate_sequences()         <- Step 1
        └── worker.generate_sequences.remote(chunk)      <- Step 2  (扇出到 N 个 AgentLoopWorker Actor)
              └── server_manager.generate(...)           <- Step 3a (RPC 到 vLLM Replica Actor)
              └── execution_pool.execute.remote(...)     <- Step 3b (RPC 到 ExecutionWorker Actor)
                    └── rate_limit_worker.acquire.remote() (RPC 到全局 TokenBucketWorker Actor)
                          └── requests.post(/run_code)  (HTTP 到 Docker Sandbox)
  └── compute_reward(_async.remote)(batch)               <- Step 4
  └── actor_rollout_wg.update_actor(batch)               <- Step 5  (发给 FSDP GPU Worker Group)
        └── self.rollout.update_weights(...)             <- Step 6  (FSDP Actor -> vLLM Actor)
  └── async_rollout_manager.wake_up() / sleep()          <- Step 6 续 (下一轮前后切换)
```

每一个箭头都是 Ray 的一次 RPC 或 Object Store 操作。这套架构没有 Ray 的话，需要自己实现 6 套通信、调度、状态管理代码。在 verl 里 Ray 把它们抽成统一的 actor.method.remote(...) + ray.get(...)，这是它在 ReTool 流程中作为“分布式操作系统”发挥作用的具体地方。

- Step 1: 触发生成 (Rollout)
    - 主控脚本（也是一个 Ray 节点）向包装了 vLLM 的 Rollout Actors (GPU) 发送指令：“开始生成”。 vLLM 开始吐出 Token。
    - 入口在主训练循环 fit() 里。一句话调用 async_rollout_manager，背后是一个 Ray 管理的 vLLM Actor 池。
        - async_rollout_manager 本身是 AgentLoopManager，它在初始化时通过 Ray 拉起 vLLM Replica：
        - Ray 的角色：把 vLLM 推理引擎包成 Ray Actor，主循环只持有一个 handle，generate_sequences 是一次跨进程 RPC。
```python
with marked_timer("gen", timing_raw, color="red"):
    if not self.async_rollout_mode:
        gen_batch_output = self.actor_rollout_wg.generate_sequences(gen_batch_output)
    else:
        gen_batch_output = self.async_rollout_manager.generate_sequences(gen_batch_output)
```
- Step 2: 触发 Agent Loop (CPU + I/O)
    - AgentLoopManager.generate_sequences 把 batch 切片，通过 worker.generate_sequences.remote(chunk) 发往多个 AgentLoopWorker Ray Actor：
        - Ray 的角色：@ray.remote 让这些 Worker 跑在 CPU 节点上；ray.get([...]) 同步 gather N 路并行。self.wake_up() 在调用前显式唤醒 Rollout Replica，调用后 self.sleep() 释放 GPU 缓存。
    - 当 vLLM 生成到 `<tool_call>` 标记时，生成暂停。 此时，数据通过 Ray 被传递给 AgentLoopWorker Actors (CPU)。
    - 如下代码：这里 ray.get 实现了 异步并发。几十个 CPU 核心同时开始解析代码，并通过 ExecutionWorker (Ray Actor) 和 TokenBucketWorker (Ray Actor) 并发请求 Docker Sandbox。 此时，GPU 可以被释放去处理其他 Batch，或者处于非常轻量的等待状态。

```python
# 将数据分块，交给多个 CPU Worker 并行处理
chunkes = prompts.chunk(len(self.agent_loop_workers))
outputs = ray.get([
    worker.generate_sequences.remote(chunk) 
    for worker, chunk in zip(self.agent_loop_workers, chunkes)
])
```

- Step 3: 多轮对话拼接，tool_agent_loop.py/sandbox_fusion_tools.py
    - 进入 AgentLoopWorker 后，每个样本跑 ToolAgentLoop.run()，里面是 PENDING -> GENERATING -> PROCESSING_TOOLS 状态机。生成是 RPC 到 vLLM Actor：
        - Ray 的角色：server_manager.generate 是发到 vLLM Actor 的 RPC；execution_pool.execute.remote 是发到 ExecutionWorker Actor 的 RPC，再被全局 TokenBucketWorker Actor 限流。一次 rollout 内部就涉及多类 Actor 联动。
    - Sandbox 返回结果后，AgentLoopWorker 把结果拼成新的 Prompt，再次通过 Ray RPC 调用 vLLM Actor 继续生成，直到模型输出最终答案。
- Step 4: 奖励计算 (Reward)，ray_trainer.py
    - 回到主循环，等所有 trajectory 完成后算 Reward。这里有两种模式：同步直接调用，或 compute_reward_async.remote(...) 异步给到 Ray 调度：
    - Ray 的角色：compute_reward_async.remote(...) 把 reward 计算丢给 Ray Task 异步执行，主循环可以并行做 old_log_prob 等步骤；如果走 RM 模型则是 RPC 到 rm_wg。
- Step 5: 模型训练 (Trainer) ，ray_trainer.py
    - Ray 的角色：actor_rollout_wg 是一组 GPU Actor（FSDP Worker）；update_actor(batch) 实际上是一次广播式 RPC + 数据传递。Ray Object Store 让大 batch tensor 能高效送到所有 Worker。
    - 所有带有 Reward 的轨迹数据被收集起来，通过 Ray 传递给 Trainer Actors (GPU, 运行 Megatron 或 FSDP)。 Trainer 根据 PPO/GRPO 算法计算 Loss，反向传播，更新模型权重。
- Step 6: 权重同步 (Weight Sync)，权重热更新（Trainer -> vLLM）
    - Trainer 显存里的权重更新完，需要同步给 vLLM 推理 Actor。update_actor 内部最终调用 self.rollout.update_weights(...)，这是从 FSDP Worker 推到 Rollout Replica 的 weight bridge：
    - 下一轮 rollout 开始前，主循环会通过 AgentLoopManager.wake_up() 把 vLLM Replica 唤醒：
    - Ray 的角色：FSDP Actor 和 vLLM Actor 跑在不同进程（甚至不同机器），权重通过 Ray 的 Plasma Object Store / NCCL 广播；wake_up/sleep 是发给所有 Replica 的并发 Ray RPC。

### ray_debug_tutorial

- https://verl.readthedocs.io/en/latest/start/ray_debug_tutorial.html